In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..') if '__file__' in dir() else os.path.abspath('..'))

from train.curricula import Curriculum, STAGES
from train.ppo import PPOTrainer
from tqdm import tqdm

In [2]:
from train.ppo import DEVICE
print(f'Using device: {DEVICE}')

curriculum = Curriculum(num_envs=64, window_radius=5, stage=0)
trainer = PPOTrainer(curriculum.num_channels, curriculum.grid_side, curriculum.num_actions)

Using device: mps


In [3]:
trainer.load('best.pt')

In [4]:
obs = curriculum.reset()
best_territory = 0

for iteration in (pbar := tqdm(range(1000))):
    buffer, obs, mean_reward, last_values, fps = trainer.collect_rollout(curriculum, obs, rollout_steps=128)
    loss = trainer.update(buffer, last_values)
    metrics = curriculum.get_metrics()

    if metrics['territory_gained'] > best_territory:
        best_territory = metrics['territory_gained']
        trainer.save('best.pt')

    stage_info = f"stg {curriculum.stage + 1}/{len(STAGES)}"
    pbar.set_postfix_str(
        f"loss={loss:.3f} rew={mean_reward:.3f} "
        f"cap={metrics['captures']} terr={metrics['territory_gained']} "
        f"avg_t/c={metrics['avg_territory_per_capture']:.1f} "
        f"surv={metrics['survival_rate']:.2f} fps={fps:.0f} {stage_info}"
    )

    if curriculum.should_advance(metrics):
        curriculum.advance_stage()
        obs = curriculum.reset()
        best_territory = 0

  1%|          | 10/1000 [00:48<1:18:25,  4.75s/it, loss=0.108 rew=1.251 cap=1438 terr=7115 avg_t/c=4.9 surv=1.00 fps=7340 stg 1/3]


>>> Stage 1 complete — advancing to stage 2


  2%|▏         | 20/1000 [02:11<2:16:04,  8.33s/it, loss=43.032 rew=0.062 cap=1687 terr=6826 avg_t/c=4.0 surv=0.95 fps=21358 stg 2/3] 


>>> Stage 2 complete — advancing to stage 3


  7%|▋         | 74/1000 [29:08<6:04:42, 23.63s/it, loss=21.629 rew=0.675 cap=5516 terr=42436 avg_t/c=7.7 surv=1.00 fps=45986 stg 3/3]  


KeyboardInterrupt: 